# Day 02 — TypedDict for LangGraph State

**Phase 1 · Module 1: LangGraph Nodes** · ~30 min

### What I practice today
- [ ] Define a **TypedDict** `State` schema with 4+ fields
- [ ] Use `Annotated[list, operator.add]` for a **list-append reducer**
- [ ] Understand: **why TypedDict (not Pydantic)** for LangGraph State

### Why this matters
In LangGraph every node receives the **state**, returns a **partial update**, and the graph **merges** that update back in. The *shape* of that state is a `TypedDict`. Yesterday Pydantic guarded the agent's outer I/O; today's `TypedDict` describes the data that flows **between nodes inside** the graph.

> Rule of thumb: *Pydantic = validate the outside world. TypedDict = describe the shared state that hops node-to-node.*

## 1. TypedDict — a dict with a known shape

A plain `dict` can hold anything — any key, any type. That flexibility is also the problem: a typo'd key or wrong-typed value passes silently and breaks a node three hops later.

A **`TypedDict`** says *"this dict has these keys, with these types."* It is **still a normal dict at runtime** (you make it with `{...}`, access with `state["key"]`) — but your editor and `mypy` now know the shape and flag mistakes as you type.

In [5]:
# A plain dict: no shape, no help from the editor
state = {"messages": [], "step": 0}
state["mesages"] = 5     # typo -> silently creates a NEW key, no warning
print(state)

{'messages': [], 'step': 0, 'mesages': 5}


Same data as a `TypedDict` — the keys and types are declared once, and wrong keys/types are caught by the editor.

In [6]:
from typing import TypedDict

class State(TypedDict):
    messages: list[str]
    step: int

# Build it like any dict -- it IS a dict at runtime
s: State = {"messages": [], "step": 1}
print(type(s).__name__, s)      # <- 'dict'
print("is a dict:", isinstance(s, dict))

dict {'messages': [], 'step': 1}
is a dict: True


☝️ `type(s)` is literally `dict`. `TypedDict` adds **zero runtime cost** — it is pure type information for tooling. That property is exactly why LangGraph chose it for state (more on that in section 5).

### ⚠️ Where's the error if you *break* the shape? — nowhere at runtime

Because a `TypedDict` **is** a plain `dict`, Python does **no** checking when you violate it. A wrong key or a wrong-typed value **runs fine and prints happily** — the bug stays hidden until some node three hops later reads a key that isn't there.

Below: a `State` says `messages: list[str]`, `step: int`. We feed it a **typo'd key** and a **wrong type**. Watch it run with **zero complaints**.

In [7]:
from typing import TypedDict

class State(TypedDict):
    messages: list[str]
    step: int

# Every line below BREAKS the declared shape -- yet all of it runs:
bad: State = {"messages": [], "step": "not-an-int"}   # step should be int
bad["stepp"] = 5                                       # typo key -> new key, silently
bad["messages"] = "hello"                             # str, not list[str]

print("ran with NO error:", bad)
print("KeyError only shows up LATER, when a node reads the real key:")
try:
    print(bad["step"] + 1)          # someone expects an int here
except TypeError as e:
    print("  TypeError:", e)        # 'can only concatenate str'

ran with NO error: {'messages': 'hello', 'step': 'not-an-int', 'stepp': 5}
KeyError only shows up LATER, when a node reads the real key:
  TypeError: can only concatenate str (not "int") to str


☝️ Runtime said nothing. So **where** is the error caught? At **static-check time** — your editor underlines it as you type, and `mypy` reports it before you ever run the code. That's the whole deal with `TypedDict`: the safety is in the *tooling*, not the runtime.

The cell below runs `mypy` on the exact bad snippet so you can **see the errors it would flag** (installs `mypy` if missing):

In [8]:
import sys, subprocess, textwrap

# Ensure mypy is available (static type checker)
try:
    import mypy.api
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mypy"])
    import mypy.api

snippet = textwrap.dedent('''
    from typing import TypedDict

    class State(TypedDict):
        messages: list[str]
        step: int

    bad: State = {"messages": [], "step": "not-an-int"}   # wrong type
    bad["stepp"] = 5                                       # typo'd key
    bad["messages"] = "hello"                             # wrong type
''')

path = "_typeddict_check.py"
with open(path, "w") as f:
    f.write(snippet)

stdout, stderr, code = mypy.api.run([path, "--no-color-output", "--no-error-summary"])
print("mypy exit code:", code, "(non-zero = it found problems)\n")
print(stdout or "(no output)")

import os; os.remove(path)   # tidy up the temp file

mypy exit code: 1 (non-zero = it found problems)

_typeddict_check.py:8: error: Incompatible types (expression has type "str", TypedDict item "step" has type "int")  [typeddict-item]
_typeddict_check.py:9: error: TypedDict "State" has no key "stepp"  [typeddict-unknown-key]
_typeddict_check.py:9: note: Did you mean "step"?
_typeddict_check.py:10: error: Value of "messages" has incompatible type "str"; expected "list[str]"  [typeddict-item]



☝️ **There** is the error — three of them, each pointing at the exact line:
- `"not-an-int"` for `step` → *"Incompatible types ... expected `int`"*
- `bad["stepp"]` → *"TypedDict `State` has no key `stepp`"*
- `"hello"` for `messages` → *"expected `list[str]`"*

Same three lines that ran silently above. **That's the trade LangGraph makes:** `TypedDict` gives zero-cost, dict-shaped state at runtime, and pushes shape-checking into `mypy` + your editor. In practice you run `mypy .` in CI (Day 30's checklist) so these get caught before the code ever runs.

## 2. The LangGraph state pattern — nodes return *partial* updates

The key idea: a node does **not** rebuild the whole state. It returns a small dict with **only the keys it changed**, and LangGraph merges that into the running state.

By default, merging means **overwrite**: the new value for a key replaces the old one.

In [9]:
from typing import TypedDict

class State(TypedDict):
    question: str
    answer: str
    step: int

# Simulate what LangGraph does: start state, node returns a partial update, merge
state: State = {"question": "What is LangGraph?", "answer": "", "step": 0}

def node(s: State) -> dict:
    # return ONLY what changed -- not the whole state
    return {"answer": "A framework for stateful agents.", "step": s["step"] + 1}

update = node(state)
merged = {**state, **update}      # default merge = overwrite
print("update:", update)
print("merged:", merged)

update: {'answer': 'A framework for stateful agents.', 'step': 1}
merged: {'question': 'What is LangGraph?', 'answer': 'A framework for stateful agents.', 'step': 1}


Overwrite is right for `answer` and `step`. But what about a running **list** — like a conversation history? Overwriting would **throw away** every earlier message. That is what reducers fix.

## 3. The problem: overwrite destroys list history

Watch a `messages` list get clobbered when two nodes each return their own list.

In [10]:
class State(TypedDict):
    messages: list[str]

state: State = {"messages": ["user: hi"]}

# node A adds the assistant reply
update_a = {"messages": ["assistant: hello"]}
state = {**state, **update_a}          # default overwrite
print("after A:", state)               # 'user: hi' is GONE -- bad!

after A: {'messages': ['assistant: hello']}


The first message vanished. With plain overwrite, the only way to keep history is to have every node read the full list and return the full list — fragile and easy to get wrong. We want *append*, not *replace*.

## 4. The fix: `Annotated[list, operator.add]` — a reducer

A **reducer** tells LangGraph *how to combine* the old value and a node's new value for a given key, instead of overwriting.

You attach it with `Annotated[<type>, <reducer_fn>]`:

```python
messages: Annotated[list[str], operator.add]
```

- The **type** is still `list[str]` (editors see a list).
- The **reducer** `operator.add` means: on update, do `old + new` — i.e. **append/extend** the list.

`operator.add` is just the `+` function: `operator.add([1], [2]) == [1, 2]`.

In [12]:
import operator
from typing import Annotated, TypedDict

# operator.add IS the '+' operator as a function
print("operator.add([1,2], [3]) =", operator.add([1, 2], [3]))

class State(TypedDict):
    # reducer attached: updates to 'messages' are APPENDED, not overwritten
    messages: Annotated[list[str], operator.add]
    step: int     # no reducer -> plain overwrite (the default)

operator.add([1,2], [3]) = [1, 2, 3]


Now simulate LangGraph's merge **using the reducer** for `messages` and overwrite for `step`. History is preserved.

In [13]:
def merge(state: dict, update: dict, reducers: dict) -> dict:
    """Tiny stand-in for what LangGraph does internally."""
    out = dict(state)
    for key, new_val in update.items():
        if key in reducers:
            out[key] = reducers[key](out.get(key, []), new_val)  # old + new
        else:
            out[key] = new_val                                   # overwrite
    return out

reducers = {"messages": operator.add}     # mirrors the Annotated[...] declaration

state = {"messages": ["user: hi"], "step": 0}
state = merge(state, {"messages": ["assistant: hello"], "step": 1}, reducers)
print("after A:", state)
state = merge(state, {"messages": ["user: thanks"], "step": 2}, reducers)
print("after B:", state)     # full history kept, step overwritten each time

after A: {'messages': ['user: hi', 'assistant: hello'], 'step': 1}
after B: {'messages': ['user: hi', 'assistant: hello', 'user: thanks'], 'step': 2}


☝️ This is **exactly** the behaviour LangGraph gives you for free when you declare `Annotated[list, operator.add]`. You never write the merge loop — the graph reads the reducer off the annotation and applies it. (LangGraph also ships `add_messages`, a smarter reducer for chat messages — same idea, handles message IDs/updates.)

## 5. Why TypedDict and **not** Pydantic for state?

Day 1 said "use Pydantic for I/O." So why switch here? Different jobs:

| | **Pydantic `BaseModel`** | **`TypedDict`** |
|--|--|--|
| At runtime it's | a custom object | a plain `dict` |
| Validation | yes, on every creation | none (types are hints only) |
| Cost per node | re-validate whole model each hop | zero — just dict merge |
| Partial updates | awkward (model wants all fields) | natural (`return {"step": 1}`) |
| Reducers (`Annotated`) | not the native pattern | first-class in LangGraph |

A graph may run **dozens of node hops**; the state is merged at every one. A `TypedDict` is a plain dict, so merging is cheap and partial updates are trivial. **Use Pydantic at the edges** (validate the request coming in / response going out); **use `TypedDict` for the state that flows between nodes.**

In [14]:
# Partial update feels natural with a dict-shaped state...
class State(TypedDict):
    question: str
    answer: str
    step: int

def node(s: State) -> dict:
    return {"step": s["step"] + 1}      # return just one key -- fine

# ...whereas a Pydantic model wants a full, valid object every time,
# which is the wrong ergonomics for per-key node updates.
print(node({"question": "q", "answer": "", "step": 0}))

{'step': 1}


### 🤔 "If it doesn't error at runtime, why use it at all?"

Fair question — and the no-runtime-check is the **feature**, not a gap. Five reasons:

1. **Speed — state hops many nodes.** A graph merges state at *every* node hop (dozens per run). A `TypedDict` **is** a dict, so merging is `{**state, **update}` — nearly free. A Pydantic model would **re-validate every field on every hop** — pure overhead on the hot path.
2. **You don't lose the safety — you move it earlier.** Editor + `mypy` catch shape bugs **before the code runs** (section 1's demo). Runtime validation catches them *during* the run — later, once the bug already reached production. Static check = fail before deploy.
3. **Partial updates need a plain dict.** Nodes return `{"step": 1}` — one key. A Pydantic model wants *all* fields valid to construct, which fights the partial-update pattern. A dict merges partials trivially.
4. **Reducers live on the annotation.** `Annotated[list, operator.add]` — LangGraph reads the reducer straight off the field. First-class for `TypedDict`, not the native Pydantic pattern.
5. **You still get autocomplete + docs.** The editor knows the shape → completion, safe refactors, self-documenting code. You just don't pay for it at runtime.

**The split — validate once, then go fast:**

| Where | Tool | Why |
|-------|------|-----|
| **Edges** (API in/out — untrusted outside data) | **Pydantic** (Day 1) | must validate — reject bad input at the door |
| **Inside the graph** (state between nodes — your own trusted data) | **TypedDict** (today) | already validated at the edge; now merge cheaply |

> Analogy: passport check **once** at the border (Pydantic at the edge), not at every hallway inside the building (TypedDict on each hop). Re-validating trusted internal data on every hop is wasted work.

## 6. Build the deliverable — a 4+ field `State`

A realistic LangGraph state for a small research/answer agent. Note which fields get a reducer and which don't.

In [15]:
import operator
from typing import Annotated, TypedDict

class AgentState(TypedDict):
    question: str                                   # input, set once (overwrite)
    messages: Annotated[list[str], operator.add]    # conversation -> APPEND
    documents: Annotated[list[str], operator.add]   # retrieved docs -> APPEND
    answer: str                                     # final answer (overwrite)
    step: int                                       # hop counter (overwrite)

# An initial state
init: AgentState = {
    "question": "What is a reducer?",
    "messages": [],
    "documents": [],
    "answer": "",
    "step": 0,
}
print(init)

{'question': 'What is a reducer?', 'messages': [], 'documents': [], 'answer': '', 'step': 0}


Drive it through two mock nodes using the same `merge()` helper, so you can see appends accumulate while scalars overwrite.

In [20]:
reducers = {"messages": operator.add, "documents": operator.add}

def retrieve(s: AgentState) -> dict:
    return {"documents": ["doc: reducers combine old+new"],
            "messages": ["assistant: searching..."],
            "step": s["step"] + 1}

def answer(s: AgentState) -> dict:
    return {"answer": "A reducer says how to merge state updates.",
            "messages": ["assistant: here is the answer"],
            "step": s["step"] + 1}

def new_appends(s: AgentState) -> dict:
    return {"answer": "A latest update.",
            "messages": ["new assistant: new answer"],
            "step": s["step"] + 2}

def new_documents(s: AgentState) -> dict:
    return {"documents": ["doc: latest version of documents"],
            "messages": ["assistant: new documents loading"],
            "step": s["step"] + 2}

state = init
state = merge(state, retrieve(state), reducers)
state = merge(state, answer(state), reducers)
state = merge(state, new_appends(state), reducers)
state = merge(state, new_documents(state), reducers)

for k, v in state.items():
    print(f"{k:10}: {v}")

question  : What is a reducer?
messages  : ['assistant: searching...', 'assistant: here is the answer', 'new assistant: new answer', 'assistant: new documents loading']
documents : ['doc: reducers combine old+new', 'doc: latest version of documents']
answer    : A latest update.
step      : 6


## 7. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **`TypedDict`** | a dict with a declared shape; editor/mypy catch bad keys & types, **zero runtime cost** |
| **plain field** (e.g. `step`) | default merge = **overwrite** — right for scalars / final values |
| **`Annotated[list, operator.add]`** | attach a **reducer** → updates **append** instead of overwrite (keeps history) |
| **`operator.add`** | the `+` function; `old + new` extends the list |
| **partial updates** | nodes return only the keys they changed; the graph merges them |
| **TypedDict over Pydantic** | state hops many times — plain-dict merge is cheap; Pydantic stays at the I/O edges |
